# PyTorch GPU Example

This notebook demonstrates GPU-accelerated deep learning with PyTorch.

## 1. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import matplotlib.pyplot as plt

# Select device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Define a Simple Neural Network

In [ ]:
class SimpleNet(nn.Module):
    """Simple feedforward neural network for demonstration"""
    
    def __init__(self, input_size=784, hidden_size=256, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.fc3(x)
        return x

# Create model and move to device
model = SimpleNet().to(device)
print(f"Model created on: {next(model.parameters()).device}")
print(f"\nModel architecture:\n{model}")

## 3. Generate Synthetic Data

In [ ]:
# Create synthetic dataset
batch_size = 64
num_batches = 100

# Generate random data
X_train = torch.randn(batch_size * num_batches, 784)
y_train = torch.randint(0, 10, (batch_size * num_batches,))

print(f"Training data shape: {X_train.shape}")
print(f"Labels shape: {y_train.shape}")

## 4. Training Loop

In [ ]:
def train_epoch(model, X, y, batch_size, device):
    """Train for one epoch and return loss history"""
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    losses = []
    num_batches = len(X) // batch_size
    
    for i in range(num_batches):
        # Get batch
        start = i * batch_size
        end = start + batch_size
        X_batch = X[start:end].to(device)
        y_batch = y[start:end].to(device)
        
        # Forward pass
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
    
    return losses

# Train for a few epochs
print("Training...")
start_time = time.time()

all_losses = []
for epoch in range(5):
    losses = train_epoch(model, X_train, y_train, batch_size, device)
    avg_loss = sum(losses) / len(losses)
    all_losses.extend(losses)
    print(f"Epoch {epoch+1}/5, Avg Loss: {avg_loss:.4f}")

elapsed = time.time() - start_time
print(f"\nTraining completed in {elapsed:.2f} seconds")

## 5. Plot Training Loss

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(all_losses, alpha=0.7)
plt.xlabel('Batch')
plt.ylabel('Loss')
plt.title('Training Loss Over Time')
plt.grid(True, alpha=0.3)
plt.show()

## 6. GPU vs CPU Benchmark

In [ ]:
def benchmark_inference(device, iterations=100):
    """Benchmark inference speed on given device"""
    model = SimpleNet().to(device)
    model.eval()
    
    x = torch.randn(64, 784, device=device)
    
    # Warmup
    with torch.no_grad():
        for _ in range(10):
            _ = model(x)
    
    # Synchronize if GPU
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    # Benchmark
    start = time.time()
    with torch.no_grad():
        for _ in range(iterations):
            _ = model(x)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    elapsed = time.time() - start
    return (elapsed / iterations) * 1000  # ms per iteration

# Run benchmarks
cpu_time = benchmark_inference(torch.device('cpu'))
print(f"CPU: {cpu_time:.3f} ms/iteration")

if torch.cuda.is_available():
    gpu_time = benchmark_inference(torch.device('cuda'))
    print(f"GPU: {gpu_time:.3f} ms/iteration")
    print(f"\nSpeedup: {cpu_time/gpu_time:.1f}x faster on GPU")
else:
    print("\nGPU not available for comparison")

## Summary

This notebook demonstrated:
- Creating a neural network in PyTorch
- Moving model and data to GPU
- Training loop with GPU acceleration
- Benchmarking GPU vs CPU performance

For real experiments, you would use actual datasets (MNIST, CIFAR, etc.) and more complex architectures.